In [ ]:
# fix relative imports
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)

# import known packages
import numpy as np
from functools import partial

from matplotlib import pyplot as plt

from tqdm.notebook import tqdm

# import adhoc packages
from qphaset.models import model_ising, model_annni
from qphaset.annni import model_annni_ext
from qphaset.exact_diag import model2ham
from qphaset.models import params_2d_lattice

## Gap analysis
Notebook for analysing the gap between the ground state and first excited state. This can be useful for obtaining the optimal path of guesses for DMRG.

In [ ]:
def compute_gaps(model_factory, l, params_extent, n):
    params = params_2d_lattice(params_extent[:2], params_extent[2:], n=n)
    gaps = []
    for xy in tqdm(params):
        model = model_factory(xy[0], xy[1], l=l)
        ham = model2ham(model).to_ndarray()
        lam = np.linalg.eigh(ham)[0]
        gaps.append(lam[1] - lam[0])
    gaps = np.array(gaps).reshape((n, n))
    return gaps

In [ ]:
params_extent=(0.01, 1.5, 0.01, 1.5)
l = 5
n = 32

gaps = compute_gaps(model_factory=model_annni, l=l, params_extent=params_extent, n=n)
plt.matshow(gaps, origin='lower', extent=params_extent)
plt.colorbar()

In [ ]:
model_factory = partial(model_annni_ext, c1=-1)
gaps = compute_gaps(model_factory=model_factory, l=l, params_extent=params_extent, n=n)
plt.matshow(gaps, origin='lower', extent=params_extent)
plt.colorbar()

In [ ]:
def compute_gaps_l(model_factory, *, l_max, pt):
    gaps = []
    for l in tqdm(range(2, l_max + 1)):
        model = model_factory(pt[0], pt[1], l=l)
        ham = model2ham(model).to_ndarray()
        lam = np.linalg.eigh(ham)[0]
        gaps.append(lam[1] - lam[0])
    gaps = np.array(gaps)
    return gaps

# TODO Reimplement with DMRG and obtain first excited state using "orthogonal_to".
gaps = compute_gaps_l(model_annni, l_max=7, pt=(0.75, 0.25))
plt.plot(gaps)